# Quantum Circuits | J5 QROM + Exact Gauge

This version composes **Vladislav Lavrentev / lightningv08's five-word loaders** with **our previously checked orthogonal polish of Ahmed Özer's v8 factors**. It updates the same public notebook and preserves the full author chain.

| Task | Selected component | Qubits | Full official development cost | Shots |
|---|---|---:|---:|---:|
| Inverse square root | mrmeows 63Q; unchanged | 63 | 108,560.358246 | 200,540,928 |
| DQI | Özer v8 Bézout–Schur; unchanged | 883 | 495,048,250,576.628 | 102,400 |
| Chemistry | Lightning J5 + our exact factor gauge | 429 | 426,972,119.371059 | 70,144 |

**Development projection: 19.50642373, not a Kaggle Public Score.** Our previous official result was 19.49787 (v13). At selection the new public leader was Lightning v15 at 19.57896. DQI statistics here use the fixed default empty salt, with 6 value and 2 phase errors and their official repetition penalty. A different grader salt can change that penalty. No salt or seed search is performed.

## Why the two improvements compose

Lightning's J5 schedule changes **how the same integer words are loaded**. Our C-space rotation changes **the factor weights** while preserving their augmented Gram. These act on independent parts of the chemistry submission. All 15 loader binaries are reused byte for byte from Lightning v15; our transformed factor NPZ is byte-identical to our v13 factor. Arithmetic and DQI hashes also remain unchanged.

J5 loads five 8-bit angle words at once instead of four. It needs 15 circuits for 75 columns rather than 19. Each uses 123 qubits, within the 128 cap, and 297 Toffolis. The whole block changes from 421Q / 21,627 Toffolis to 429Q / 20,961 Toffolis; the complete cost includes the extra output storage. This loader design and its 75-relation kernel budget belong to Lightning and the credited upstream compiler authors, not this notebook.

Our exact gauge further reduces the J5 chemistry cost from **426,973,779.381136 to 426,972,119.371059**, a **0.000389%** controlled gain. The gain over Lightning is small. Most of the improvement over our v13 comes from his J5 loader schedule.

For each group A = [w; w_B], right orthogonal rotation A → AO preserves AAᵀ, the Hamiltonian, one-body contraction and offset norm. The physical errors and SOS gap are preserved, while the L1 normalization can decrease. We find pair rotations on Mac MPS using sorted sign-change knots and prefix sums: endpoint and analytic stationary-point comparisons give the pair minimum in O(B log B), with disjoint pairs processed together. Gauge rotations themselves are prior public work. After float64 polar projection, our orthogonality error is 5.6e-15 and augmented-Gram drift 1.4e-14.

Official physical metrics: error₁ **0.01891123452850** Ha; error₂ **0.01999718500074** Ha; E_gap **3.91509479776505** Ha. Lambda_eff drops from 47.482318353814 to 47.482133749675. All words, tree edges and unrelated factor fields stay fixed.

## Independent checks and reproducibility

Both Lightning's baseline chemistry and our composition passed the unmodified full official native verifier independently on Kaggle CPU, 70,144 shots each, with original random XOR targets and measurement outcomes. The unchanged arithmetic and DQI full reports are reused only for their identical circuit hashes and unchanged official/native fingerprint; no reports are borrowed for changed circuits.

We also simulated the **actual 15 published J5 gate streams on MPS**: every 1,096 promised address, zero and nonzero output target patterns, and two measurement streams. All 65,760 trajectories passed input restoration, differential-XOR output, scratch clearing and exact phase repair, in 81.35s. The check uses the official contract f_k = theta_k XOR theta_(k-1); the loaders are applied sequentially to obtain each next batch. It is not a claim to enumerate all 2^40 arbitrary targets or replace native randomized checks.

Run All authenticates all 18 source members, reconstructs the selected factor byte for byte, validates full-check receipts against official Python/native/reference fingerprints, reevaluates physical accuracy, and checks every ZIP member after compression. A changed native/verifier fingerprint triggers full checks; FORCE_FULL=True reruns them explicitly. The exact saved float64 gauge avoids SVD-replay differences. The compiled J5 files are reused from the pinned public output; this notebook does not claim to rederive their compiler or the upstream unpublished fitters.

Outputs retain Lightning's compiler source, plans, LICENSE and NOTICE as provenance, along with our MPS source, decoded gate streams, factor, reports, certificate and archive manifest. Kaggle accelerator is **None / CPU**, Internet off. Saving a version and creating a competition submission are separate states; only the graded submission establishes a new official score.

## Sources and credits

- [Vladislav Lavrentev / lightningv08, Bezout-Schur + 5-Word QROM v15](https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom?scriptVersionId=362616133): J5 schedule, relation-budget tuning, exact compiler/certificates and resource comparison. Its 15 binaries are reused unchanged, and all original source/notices are retained.
- [Ahmed Berat Özer v8](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression?scriptVersionId=362606565): Bézout–Schur splice, B255 dictionary, pruning, quantization, offset/refit/gauge loop and generalized kernel-relation compiler. [v8 components](https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components).
- [mrmeows open circuits](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits): 63Q arithmetic, round17 Bézout–Hankel transform, paired-product lookup algebra, kernel tools and HMR emitter; marculera's four-address loader lineage.
- [Alex Chilton's Schur rebuild](https://www.kaggle.com/code/alexchilton/toffolis-go-18-3-schur-dqi-at-883-qubits), based on [jday96314 / QuantumAI](https://github.com/jday96314/QuantumAI): Schur architecture and 883Q synthesis. The original repository has no declared code licence; original rights remain retained and the public compiled component is credited.
- Inherited lineage: SomeJoint / Agniv Banerjee, Dmitrii Gluzdov, ViberTeam / Artem1410 / anazemcev, manwithacat, Phuc Thai / DePie, Kozykorpesh / kozykappa, Michael Rustad, busyaprime, marsyas, Sumit Chongder, Rudra Bhaskar, Zongao Bian, Mizero Lucky Gall, Gu et al. and Guang Hao Low.
- Kirill Tymchishin: prefix-sum MPS pair minimizer, earlier exact factor polish, this J5 composition, actual gate-stream MPS checks, independent complete chemistry checks and integration. Earlier original work remains in the version history.

Our integration/gauge code is Apache-2.0. Original component rights and licences remain with their authors; official inputs and verification software keep Google Quantum AI / competition terms.


In [ ]:
"""Rebuild the credited public portfolio and apply a certified exact C-gauge.
Author of integration and prefix-sum gauge search: Kirill Tymchishin.
Upstream circuit and factor authors retain their rights; see NOTICE.txt.
"""
import os
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1')
from pathlib import Path
import sys,json,hashlib,zipfile,shutil,base64,zlib,io,gc,time
import numpy as np
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working');SUB=WORK/'submission_files'
BASE_PINS={'invsqrt.kmb': '9ac50386192a544f353ab0f46604e5e072981b8bfc8521e8551c689a9445e3b9', 'dqi.kmb': '569863ae0fafd5f94fe91b498ec67ed0a4d96b826ced31cb0ff57c7fe6c063cb', 'dfthcsos/angle_k_14.kmb': 'f3bb8ce79e39ecddad5bb1ff57e6818158c4cba1133b54f0009f1133b8585d49', 'dfthcsos/angle_k_3.kmb': 'ac59a43cf75a516937ad12900916c292add65558902df77b18d0600cc45a7ecb', 'dfthcsos/angle_k_11.kmb': '9b103b5aadfc52e1b481894dff426dbf7443b5a3530db2c1e325ef6a39ba63e2', 'dfthcsos/angle_k_13.kmb': '42044bcb48d80947ef2900aea4eb2eaa8593f1b809d37bada48f1ab169b44ae3', 'dfthcsos/angle_k_10.kmb': 'ac3cdb0d8d742d9629d3d5e3d5acb843012a084557587fe2c0ad9b9058da06d5', 'dfthcsos/angle_k_9.kmb': '039ec2a8c81ff647a83832dc76a687947a80f81202361b9af2f1c6cef1e29268', 'dfthcsos/submission.npz': 'a5eabd76c2b2869e39511a53130cd909034dec6f26fa3db61354a9d12ffe6413', 'dfthcsos/angle_k_12.kmb': 'd872682a1672429fdbbb44f56dbac9f4a0e951231817926d5470ff36f64aac3d', 'dfthcsos/angle_k_6.kmb': '9932019836bbe8217a8fd34d1b54bbaccf9591d1c83f56c2a0cb20e2e14a2c7d', 'dfthcsos/angle_k_8.kmb': 'd748449d2c77cf858bdd90d69054bc3a3ac1d81ad2501c21d85e5b04a824e66b', 'dfthcsos/angle_k_2.kmb': 'e25d78895511c50babe940b21314ede6ad79794d38a9da53efaefbdc802864ac', 'dfthcsos/angle_k_5.kmb': '6c4b1e3a59e46a604528f80912f4ccfd9ce689c0f39cd548521d0663d9d1821a', 'dfthcsos/angle_k_4.kmb': '64d9dfc71e0fb4c5c55e4a01ba9063c579d17d2602551a0899253f8edd1dc136', 'dfthcsos/angle_k_1.kmb': '974dc12398186f359e79d38d733720c5231686d7f591dcb1ac1b5d20be27f47e', 'dfthcsos/angle_k_7.kmb': '7f1444666cf99cac8533089c862edfeb60db5e41daedf693c03af0d0be4b4392', 'dfthcsos/angle_k_0.kmb': 'b631c3cafd2edc600edeed997f28c1b1b8a71234189bcc7d6a3afdf4a47408db'}
CERTIFICATE={'task_reports': {'invsqrt': {'status': 'PASS', 'score': 108560.35824641786, 'toffoli_equivalent': 1723.1802896256804, 'toffoli_count': 1404.0938657768652, 'max_toffoli_count': 1490, 't_count': 1.0, 'rotation_count': 44.0, 'rotation_eps': 1.8534217822524296e-13, 'num_qubits': 63, 'reaction_depth': 785, 'num_ops': 41252, 'verified_shots': 200540928, 'worst_error_over_tolerance': 0.9996626562985852, 'worst_x': 60, 'circuit_sha256': '9ac50386192a544f353ab0f46604e5e072981b8bfc8521e8551c689a9445e3b9', 'seconds': 401.62666370899996}, 'dqi': {'status': 'PASS', 'score': 495048250576.62756, 'm': 4095, 'n': 70, 'degree': 12, 'r': 2016, 'verified_shots': 102400, 'value_errors': 6, 'phase_errors': 2, 'empirical_error_rate': 7.8125e-05, 'effective_error_rate': 0.0011344950381555927, 'log2_delta': -128.0, 'repetition_factor': 1.2230139975925918, 'mean_error_weight': 31.046737331746634, 'toffoli_count': 519152.19590820314, 'max_toffoli_count': 519157, 'reaction_depth': 10267, 'num_ops': 88933502, 'shot_operation_counts': {'NEG': 307054, 'BIT_INVERT': 250209176629, 'BIT_STORE0': 190260199452, 'BIT_STORE1': 674028403, 'X': 15968870400, 'Z': 13926183516, 'R': 7733677661, 'HMR': 46804885109, 'CX': 7084735711390, 'CZ': 116936216011, 'SWAP': 169239520596, 'CSWAP': 1262796800, 'CCX': 51898388061, 'POP_CONDITION': 271558348800, 'PUSH_CONDITION': 271558348800}, 'num_qubits': 883, 'num_bits': 169120, 'circuit_sha256': '569863ae0fafd5f94fe91b498ec67ed0a4d96b826ced31cb0ff57c7fe6c063cb', 'seconds': 214.616666555}, 'dfthcsos': {'status': 'PASS', 'score': 426972119.3710589, 'lambda_eff': 47.48213374967529, 'toffoli_count': 20961.0, 'num_qubits': 429, 'brot': 8, 'batch_size': 5, 'num_circuits': 15, 'reaction_depth': 165, 'num_ops': 628250, 'verified_shots': 70144, 'submission_sha256': '6e25379c221b415ff9003d138bf0c96c2f0f32214c92492983cccdedc154e8e1', 'resources': {'brot': 8, 'keep_bits': 10, 'angles_toffoli_per_pass': 4455.0, 'angles_toffoli': 8910.0, 'angles_qubits': 123, 'angles_selection_qubits': 11, 'angles_persistent_qubits': 40, 'angles_temporary_qubits': 72, 'outer_state_prep_toffoli': 378, 'inner_state_prep_toffoli': 9423, 'givens_rotations_toffoli': 2250, 'outer_state_prep_persistent_qubits': 42, 'outer_state_prep_temporary_qubits': 96, 'outer_state_prep_qubits': 138, 'inner_state_prep_persistent_qubits': 43, 'inner_state_prep_temporary_qubits': 304, 'inner_state_prep_qubits': 347, 'heuristic_persistent_qubits': 85, 'heuristic_temporary_qubits': 304, 'heuristic_toffoli': 12051, 'heuristic_qubits': 389, 'persistent_qubits': 125, 'temporary_qubits': 304, 'toffoli': 20961.0, 'qubits': 429, 'angles_within_cap': True}, 'metrics': {'error_1': 0.018911234528504562, 'error_2': 0.01999718500074451, 'Lambda': 289.8883820122711, 'E_SOS': -22144.188444797765, 'E_gap': 3.9150947977650503, 'lambda_eff': 47.48213374967529, 'gap_in_domain': True}, 'seconds': 24.30832353699998}}, 'dependency_fingerprint': 'f86fd0ae423e599120601e32f0c65e51c14e40c171fe81928f5e103fdae391ab', 'conditions': 'Unmodified full official chemistry verifier, empty salt, seedNone; 64shots per all1096addresses, random arbitrary XOR targets and measurement outcomes. Unchanged invsqrt/DQI reports reused only for identical exact hashes and native/verifier fingerprint.', 'gauge': {'before_cost': 426973779.3811362, 'physics_before': {'error_1': 0.01891123452850644, 'error_2': 0.019997185000744593, 'Lambda': 289.8906208892117, 'E_SOS': -22144.188444797765, 'E_gap': 3.9150947977650503, 'lambda_eff': 47.48231835381439, 'gap_in_domain': True}, 'physics_after': {'error_1': 0.018911234528504562, 'error_2': 0.01999718500074451, 'Lambda': 289.8883820122711, 'E_SOS': -22144.188444797765, 'E_gap': 3.9150947977650503, 'lambda_eff': 47.48213374967529, 'gap_in_domain': True}, 'max_gram_drift': 1.4210854715202004e-14, 'orthogonality': 5.551115123125783e-15}, 'mps_j5': {'device': 'mps', 'status': 'PASS', 'scope': 'Actual J5 differential-XOR gate streams, official f_k=theta_k XOR theta_(k-1), including HMR and conditional phase repair; all1096promisedaddresses, two target patterns and two measurementstreams; arbitrary-target general correctness separately tested by full official native verifier.', 'basis_trajectories': 65760, 'batches': [{'batch': 0, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 1, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 2, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 3, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 4, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 5, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 6, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 7, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 8, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 9, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 10, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 11, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 12, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 13, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}, {'batch': 14, 'status': 'PASS', 'addresses': 1096, 'cases_per_address': 4}], 'seconds': 81.35430629202165}, 'baseline_chemistry_full_report': {'status': 'PASS', 'score': 426973779.3811362, 'lambda_eff': 47.48231835381439, 'toffoli_count': 20961.0, 'num_qubits': 429, 'brot': 8, 'batch_size': 5, 'num_circuits': 15, 'reaction_depth': 165, 'num_ops': 628250, 'verified_shots': 70144, 'submission_sha256': '807534d81230861716d26f8f7643a4cb6a028a3f7856a44fab9442aaea17c37b', 'resources': {'brot': 8, 'keep_bits': 10, 'angles_toffoli_per_pass': 4455.0, 'angles_toffoli': 8910.0, 'angles_qubits': 123, 'angles_selection_qubits': 11, 'angles_persistent_qubits': 40, 'angles_temporary_qubits': 72, 'outer_state_prep_toffoli': 378, 'inner_state_prep_toffoli': 9423, 'givens_rotations_toffoli': 2250, 'outer_state_prep_persistent_qubits': 42, 'outer_state_prep_temporary_qubits': 96, 'outer_state_prep_qubits': 138, 'inner_state_prep_persistent_qubits': 43, 'inner_state_prep_temporary_qubits': 304, 'inner_state_prep_qubits': 347, 'heuristic_persistent_qubits': 85, 'heuristic_temporary_qubits': 304, 'heuristic_toffoli': 12051, 'heuristic_qubits': 389, 'persistent_qubits': 125, 'temporary_qubits': 304, 'toffoli': 20961.0, 'qubits': 429, 'angles_within_cap': True}, 'metrics': {'error_1': 0.01891123452850644, 'error_2': 0.019997185000744593, 'Lambda': 289.8906208892117, 'E_SOS': -22144.188444797765, 'E_gap': 3.9150947977650503, 'lambda_eff': 47.48231835381439, 'gap_in_domain': True}, 'seconds': 25.577880645999983}}
ORTHOGONAL_B64='...'
EXPECTED_NEW_NPZ='5e3a787c921924aa7ed7523ed3a32c54b6a06da53905ba2bd117a8d671d96e6e'
NOTICE='''Quantum Circuits | Schur DQI + Exact Gauge
Kirill Tymchishin: independent MPS search, sorted sign-flip prefix-sum pair minimizer,
parallel perfect-matching sweeps, exact orthogonal chemistry polish, independent
full official verification and reproducible integration. This version updates the
same public notebook; previous exact-control rewrites remain in its history.

The portfolio combines the following credited public components:
- Vladislav Lavrentev / lightningv08: J5 loader schedule, 75-relation kernel
  budget, exact source rebuild and resource comparison; public v15 binaries reused.
- Ahmed Berat Ozer: v8 Bezout-Schur splice and B255 chemistry with fine
  quantization, pruning, offset optimization and refit/gauge loop.
- mrmeows: 63-qubit structural inverse-square-root build, chemistry factor family,
  round17 Bezout-Hankel transform, exact loader compiler, eight-bit regime and
  gauge refinements; marculera loaders.
- jday96314 / QuantumAI: continued-fraction Schur decoder architecture and original
  implementation. Alex Chilton: rebuild and 883-qubit reverse synthesis. We reuse
  the exact public compiled binary, not claim a new decoder or reproduce its code.
  The upstream GitHub repository has no declared code licence; its original rights
  are retained. The compiled component is sourced from the credited public output.
- Earlier gauge/refit lineage: Kozykorpesh / kozykappa, Phuc Thai / DePie,
  anazemcev / Artem1410, Dmitrii Gluzdov, Michael Rustad, Ahmed Berat Ozer, mrmeows,
  Alex Chilton. Earlier arithmetic/decoder lineage: busyaprime, SomeJoint / Agniv
  Banerjee, manwithacat, marsyas, Sumit Chongder, Rudra Bhaskar, Zongao Bian,
  lightningv08, Mizero Lucky Gall, Gu et al. and Guang Hao Low.

Sources:
https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom?scriptVersionId=362616133
https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression
https://www.kaggle.com/datasets/ahmedberatozer/vqcc-v8-components
https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits
https://www.kaggle.com/code/alexchilton/toffolis-go-18-3-schur-dqi-at-883-qubits
https://github.com/jday96314/QuantumAI

Our gauge/integration code is Apache-2.0. All original component rights and licences
remain with their authors. Official inputs, native library and verifiers retain
Google Quantum AI / competition terms. No endorsement is implied.
'''

def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
COMP=next(p.parents[2] for p in INPUT.rglob('verify_submission_invsqrt.py') if p.parents[2].name=='google-verifiable-quantum-circuit-compilation')
sys.path[:0]=[str(COMP/'kickmix'),str(COMP/'verifiers')]
import kickmix as km
from invsqrt import verify_submission_invsqrt as IV
from dqi import verify_submission_dqi as DV
from dfthcsos.verify import verify as CV,load_inputs,evaluate_words,passes_accuracy

# Select by complete member hashes, not by an unpinned "latest" filename.
sources=[]
for p in INPUT.rglob('submission.zip'):
 if '5-word-qrom' not in str(p):continue
 with zipfile.ZipFile(p) as z:
  names=[n for n in z.namelist() if not n.endswith('/')]
  if set(names)!=set(BASE_PINS):continue
  if all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in BASE_PINS.items()):sources.append(p)
assert len(sources)==1, f'Attach the pinned Lightning v15 J5 output: found {sources}'
# Retain the exact compiler sources and original licence/notices as provenance.
up=WORK/'upstream_j5';up.mkdir(exist_ok=True)
for p in sources[0].parent.rglob('*'):
 if p.is_file() and (p.suffix in ['.py','.json'] or p.name in ['LICENSE','NOTICE','README.md']):
  if p.name=='__output__.json':continue
  dst=up/p.relative_to(sources[0].parent);dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(p,dst)
shutil.copyfile(COMP/'verifiers/LICENSE.txt',WORK/'LICENSE.txt')
SUB.mkdir(exist_ok=True)
with zipfile.ZipFile(sources[0]) as z:
 for n in BASE_PINS:
  assert not Path(n).is_absolute() and '..' not in Path(n).parts
  p=SUB/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(n))

# Use the saved float64 orthogonal matrix, avoiding hardware-dependent SVD replay.
O=np.load(io.BytesIO(zlib.decompress(base64.b64decode(ORTHOGONAL_B64))))
assert O.shape==(4,128,128)
assert np.max(np.abs(O.swapaxes(1,2)@O-np.eye(128)))<1e-12
np.save(WORK/'orthogonal_gauge.npy',O)
with np.load(SUB/'dfthcsos/submission.npz',allow_pickle=False) as zz:fac={k:np.array(zz[k]) for k in zz.files}
A=np.concatenate([fac['W'],fac['H2_I'][:,None,:]],axis=1);A1=A@O
assert np.max(np.abs(A1@A1.swapaxes(1,2)-A@A.swapaxes(1,2)))<1e-10
new=dict(fac,W=A1[:,:-1,:],H2_I=A1[:,-1,:]);new['offset_W']=new['H2_I']-new['W'].sum(1)
np.savez(SUB/'dfthcsos/submission.npz',**new)
assert sha(SUB/'dfthcsos/submission.npz')==EXPECTED_NEW_NPZ, 'Rebuilt factor bytes must equal the fully verified candidate'
for key in fac:
 if key not in ['W','H2_I','offset_W']:assert np.array_equal(new[key],fac[key]),key

# A cached full check is valid only for the exact selected files and unchanged
# official Python/native sources and reference assets. Otherwise rerun in full.
h=hashlib.sha256()
for p in sorted((COMP/'verifiers').rglob('*.py'))+sorted((COMP/'kickmix').rglob('*.so')):
 h.update(str(p.relative_to(COMP)).encode());h.update(p.read_bytes())
assert sha(COMP/'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')=='7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87'
assert sha(COMP/'verifiers/dfthcsos/FeMoCo76o113e.fcidump')=='e5a48af746fd075026dec9e8f5df2d363802a121e8c53af9a0b984bde695e1bd'
FORCE_FULL=False
reports=CERTIFICATE['task_reports']
if FORCE_FULL or h.hexdigest()!=CERTIFICATE['dependency_fingerprint']:
 reports={}
 for task in ['invsqrt','dqi','dfthcsos']:
  start=time.monotonic()
  if task=='dfthcsos':r=CV(SUB/task)
  else:
   c=km.Circuit.from_file(SUB/(task+'.kmb'))
   r=IV.verify(c,IV.ProblemSpecInvSqrt.for_n_p(10),max_workers=4) if task=='invsqrt' else DV.verify(c,DV.ProblemSpecDQI.benchmark_instance(num_shots=102400),max_workers=4)
   del c;gc.collect()
  assert r['status']=='PASS',r
  assert r['verified_shots']=={'invsqrt':200540928,'dqi':102400,'dfthcsos':70144}[task]
  reports[task]=r;print('Fresh full check:',task,r['score'],time.monotonic()-start,flush=True)
else:print('Exact-member full-check receipts authenticated; official dependency/native fingerprint unchanged.')
for task,r in reports.items():
 assert r['status']=='PASS' and r['verified_shots']=={'invsqrt':200540928,'dqi':102400,'dfthcsos':70144}[task]
 print(task,'FULL PASS receipt:',r['verified_shots'],'shots; cost',r['score'])

# Re-evaluate all physical metrics on every Run All; loaders remain byte-identical.
physics=evaluate_words(load_inputs(new),new['angle_words'],new['tree_edges'],8)
assert passes_accuracy(physics)
for key in ['error_1','error_2','E_gap','lambda_eff']:
 assert abs(physics[key]-reports['dfthcsos']['metrics'][key])<1e-8,key
members={str(p.relative_to(SUB)):sha(p) for p in SUB.rglob('*') if p.is_file()}
expected=dict(BASE_PINS);expected['dfthcsos/submission.npz']=EXPECTED_NEW_NPZ
assert members==expected
out=WORK/'submission.zip'
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for n in sorted(members):
  entry=zipfile.ZipInfo(n,date_time=(2026,10,27,0,0,0));entry.compress_type=zipfile.ZIP_DEFLATED
  z.writestr(entry,(SUB/n).read_bytes())
with zipfile.ZipFile(out) as z:
 assert z.testzip() is None
 assert {n:hashlib.sha256(z.read(n)).hexdigest() for n in z.namelist()}==expected
projection=(2076774.233715282/reports['invsqrt']['score']*24439474177519.598/reports['dqi']['score']*3355599588.3155365/reports['dfthcsos']['score'])**(1/3)
receipt={'target_notebook':'kirilltym/quantum-circuits-qrom-tower42','public_score':'Not established until Kaggle grades this exact output','projected_score_not_official':projection,'archive_sha256':sha(out),'member_sha256':members,'task_reports':reports,'physics':physics,'dependency_fingerprint':h.hexdigest(),'source':str(sources[0]),'verification_conditions':CERTIFICATE['conditions']}
(WORK/'verified_release.json').write_text(json.dumps(receipt,indent=2))
(WORK/'NOTICE.txt').write_text(NOTICE)
(WORK/'gauge_certificate.json').write_text(json.dumps(CERTIFICATE,indent=2))
print('Archive:',out.stat().st_size,'bytes;',len(members),'authenticated members;',sha(out))
print('Development projection, NOT official Public Score:',projection)
print('Controlled chemistry cost:',CERTIFICATE['gauge']['before_cost'],'->',reports['dfthcsos']['score'])

# Export Linux-native decoded gate streams for standalone Mac MPS reproduction.
mp=WORK/'mps_inputs';mp.mkdir(exist_ok=True)
with zipfile.ZipFile(sources[0]) as z:(mp/'base_factor.npz').write_bytes(z.read('dfthcsos/submission.npz'))
streams={}
for p in sorted((SUB/'dfthcsos').glob('angle_k_*.kmb')):
 c=km.Circuit.from_file(p);q=mp/(p.stem+'.kmx');c.to_file(q,format='kmx');streams[p.stem]=q.read_text()
(mp/'angle_kmxs.json').write_text(json.dumps(streams))

RESEARCH_B64='eNrtO2lz2ziyf4WbqYSkDNG8D8lMreNkZrK7iV1JdmfqSRoVREIWY4qkScpHUnm//XUDpET5SOyqmbU+vKRik0Cj0Rf6QDNfny2LavrZmUYLFp1pxfWzgfRszP++O/koFTQ6Y7E0o1VSSXVJP7OozsuEVVI+l+oFk2hUr2gqFatZmlQLAP2HI6U5jVlZaePsME2lC5omsUTjuGRVxaqh9IWV+T4tZwngK6+l348/SDUtT1kNc/VlLi0ZrVYlW7Kslqq6ZHSJqD7BZlG+LFJWM2mVRQuancJ2Ga2TCyZdsDKZJ6yUgMyKFbSkNYNFgpFxliyLvKylvBpneaWx7CIp82wkH787mb7/97vpp18/vDl8/VGehLIhr6GB0WhBstWyuJZoJWUF+VzlGSkZqZMlG2fzMl+CgOpFmsykZs0JvDYzUZ6mIK0kz6p29ihfZTUrxxlHrVWsngL6ab0AJuNKMdQhBRG1W2szFH4WVxqoSEuqKb2gSUpnKVPUYR0iEdoyz/I6z5IIhmJ2EcoAKg/LPK9DJEWZTudJyqZTVQPp5+kFLNVAOiDafQSdJlmxqmHFu8OP/wwV4+DAtdS+Mc4uk3oBHGuoSgXR7ctgA2w6p6h/LSu+yCoK5cvgMi/jKhQEI99TLjDly0gGBcHWfF6eaFEOo6qq1bkChJK4vi5YsyzJatdWX5iOAxZDQG6rZVaFfKFWLWjBhu/Dw549/C1U3u+5prq/71qgpfW2oG2wBeW3nmsRwJ1ELLx7CxQC4lN+I8DnEG0yBDTPD4cRMFfh8/7+4fCSJaeLusWeZ6xSfoT54GCLlB+Bj7OYzfnhUuAYVOqgZPWqzCT+1mv2V7VqtVT6BkBXLG3WVzUuGvGlCjLw8uVn9YWhSvO8lD5LSSYJCgxDncBCcbC214p1PcOw9jjbvcDw3L3PPc+wuBLUnsLH78DqdJBOkda7qBLTo8+Tly9n6p1I+MhsM+JztOKshwIJp+DlSwMRDFMKOrih7B8J+Od//+tfrVmu0lSBFSpBM//RwoJeo9WHeNi5/VdKcwKEQZ8tryoNJ2W0JxpPa3ZVg2UPSxbxszCagAcABs862nDUwTiTEDJs8Lfn42wq7wHnypk6GaYJ5xOgtKpIk5q/w8k+b0hEzwnUGKZFflO/z8fwfDQwjEkIlgOPhjFw4KWjuCHX3tdvw2IBkt7C/wPRDoHJJKNpqA/zQnDL9YnEIsecaORWoiE+C1YUFUeSuQT+SqISwNORPkH4kXx4cvLm/evpp+Pphze/vP346c0Hmcidx7fvfz6efjr8RZ4Mojyrk2zFEFlehIhjeA5EAGnK1cgYTIRtXSFiiu+4Zcm4CSxpHS2UUj4fj+M9mVyBwAFdDEgosBazK0VO5rK616zCF45GYmnFpNEE2dVoUYBLVpS8IOcVwfWEqshbEurw83IB7lZKDlKWAUjFtS51YUMYHSUTLp4IQ1eIhtoKBmHEdjiqRSk4HyG59YlBkIFY+iJEJY5mk0a0IJBQPvpdlihgadFxCvCpLvM0PK9QYk24DUfwakwmw2QvNDjYTfI5JkExrLuBvRk3J10oQwA2+4nN2/DeCm8DakzUzeZrqi44txckQ36bmNl4FXCKSc2WcCi4yJ6bnHd8bubbHVufV7OsyterO5aNJ6SuJ3+E56OG2MnoPYibDCZribWW1sr2d3lwPuIihHVCBzjN0o7wBYQx4ZgF7Is7Qdew5hZsu/zORb+++yALDptDCLIbxiW9DLmLhADo2oblmqbl264XWLru7IGtCtieYdumGzi6rfu+H7iWrXvqC3SJKsfxR4g/wed6ndeDA9PanhWvL0Jc2DwKz83pmoUZusOrHx/DWXsM1WFjxnwH4ZA6AuGD7Rs/ZF0p/o88uLngTglWbFDSBA7Wf2i6Ym/KEowCD7E4vFJjhDFLayoyj9GAnPWcAfzbcyZ/KO2YcgYBWYxzps7Ece34z2manLE1/MCZ8B3YVQHJIIsbH/wH32mEgXidlADiyVaGQhxI1li5XNVMMYlOjC6ie8NvC/D4ACxtZZ/sHBJ7RYQRAmFEBd6J3OTx8r3gItSQLTLFUkz081UNCWd3NXqS85EDNqLRDDJEfr4VsaKKSjSWm+Bc4beh+TCHbSJx62++Qu6KaAaIsqb1qpIH8snhx4/ymh8GQ4dE5lnHFGQ+bfkc2N/UYVFifGmWnJF5uqoW4adyxdQ2lccMvbqGqgTqiuQLJuglw5Q//CoLlwM7Ymp+m4AKEmOcPRSFFJRPcTKfM0zRE5r2UWinUMy0lRCBwmueRDAlzadnIZRgNZ2eiSKKP3MDJaDaKF3BsT+VwGdwB43BIMFyBMs1FBWIqaBJOZRomhp64BaQvcMJidciIbwcEwaLhQ64YahlENeNMq2hDTC1VV2/WXXKMlbChqAPUEoNPmpTn6XXkBBVNZaX1xJ6hg1rN6o6DeTES9BptwSVB+97hkOEcvG1UTvIlCGzMHKzRurX34ZNMrdV9jbp3GUJ5iTyOZ79xVDKVIrQJMEUIatDU23t4RaIum0Zz4j07NYuWF5/RY8zfiYMYwwjY4QbPyNiXFiIGEcj2UygqYjx/7eW71jLWmC3bQal5zqeq68huO3g8EhkDl/Fr/UkTgloMXiPdsTkWhg4j1LqzN3yLQhji/lv5P7Njafc3HzKza2n3Nx+ys2dp9zcfcrNvafc3H/KzYMndTJP6+Ke1McZT+rkjCf1csZf7+bw16TNYERixs+aoVmObemuGZi6abjOOPvWZkyndHXKphWjmFp3WxJvrqCmkzAPkZZJliyTLxRzFCxkjpVXUpqfSq8gncGMIU1ZCslR9TmHXI0vqSSAfHfyURtnJ2UCBdBRn280kP6Zf7k+y8sCai8iLcslyy8hnTlM2ZV0tEhSSB7hbbGE3OMYSjwinSxWkfRpQRNAdbwq8Xo/5amNoAYypZRGrJKgJILqtU4iaVbmNAYx1RK/Wa8gWwRsVXKa9edpUkhFyebJlVSteK/jr2hbYCpMoPZeEMi1QEDVdzsYf0KDQtRFNIM8FVTJYsV0Dd10ITGmRdjSoB2WpyuU3Am+lbCOFhoY0ZQ244rc77cJPuF3kaBOErM5XaWQjNt3LqhLSAvvgr8bvLpkrLgPvDytQljDqcVVeB8LB6W8owGzabvIcAi49e7nUW26F77cbc8UZoiK0Ipk3wRl815AXtRTNFKFkpk6kH6C5Lik15U00jSNlGCPeCtV0abgv1xAyq/Qv4U6oRpaEZTAs+YByJttgc0QrJ0l/XYBFspnQHx7iQDJOcUap1Sai/aaZqYyI1QlhakOu6ACLx95qRP+m2yu2sUFiBgtsFwSj1CasTLkzxrav+hsSFd4aRIqFe1VM/UAr3fibS45AOmbPQChRNdAkrN7AWYCINoC+N8NxKxFEd8PwXGM8aaVzeeb++2KxPy6WuF0xCAWhW8Yz/AJEceReIKxWJ3wezpwUnVYNY0ccgZ2FidLUSAOOf72jkJQE9FaGeEagj/2Yg2KOSAQ13LxqVoEHkJ0hSYEfiCdh+QVOSKvQ45vmOZhB9eteykuf25UA2OiCt1xTMNF0l34HaW2q1G5fCnKKgqVw97h3uve6/6r3qv+Ue9I3TeHVRUe8rfXeGbaO1lhWVVFogiA9goTTXszLVSi4MjLMM3VF/zxIFwkKsEnAmP8rg5OzQwrOyj+lHrTQzvs1VqUwzndewVPVYK2rlXgiwGruqe8bqeBrFvTXO3V2tC5KDabwMZk84b0bN6QMiGOIW/sbKFIc7JIOPGtwJL4KuQ7aeBTIJbdNg9eoIcCV8cOYCEnl7EvrDlCP0m/APWxNE9zWlumFNEsYmkqghE9hVMNcWdO8Z4yA6/UB3sreFS8hBkIN1KexiHV6AyE0vYcNwKb3TMB9isuFIQ0uVXgBfpkWCXNBBfuZgJ2ythlqNBelO/NelWi3renMgOQPv0OCL/l4/ruGg2gPwBm+gbre4TTQG6dAD6MB2ec/QThnUVnEhNphYjBTWLByrXoYpaB6EATq5SW0mmZxARvQTbpROuwSprFmWKZxDW2Og2z7ryggoKZjAZEn4T6cAYPBj4IlW+FgnGG+zXrUyCngMxC0eHsEVO3g61tOJ0gXrx2FhLv4eLmMMxuDjemv5HvtgbuxNKnj8Eyzq5R210radaC+rsm0ozeT8xdOPr0EThASSJzmeV5qijXByGXliaOnqpdYFug2tM13QIMabq+Wl7f/oKqpycf3vz89vfpu7fv377DBOzNx09TcY+7dfP3gK8nCOyRX06LJDpLWfgzHs7mgwrsIt75PcVv3/+Kojn94MjvXP2rOX37MATjbLZpCXP3dUkWjXjJYDIh4OI+YMwJEa75QqOR7tFzM8QejQj2aVLViugwHGGDPF9lnf54suk/HIEKkHEBsW6ybrXwOMrR4Gh/35x0zZ5sQYkt2QWDdC1u1uCSwUTtdsxVlfexgcaRgNEnIsKO+sZksicejQG8iPSMm4dytQ4yV123ZG7MbPPJBq4qc/D4sIwcE5qQWUJqziU6zi1TBqlyp7llyaIRiYL6wvMOwDJp2upf+CGkyaTtEIOHEWOzztgaKlw73OEaKuTedQ/9LFQOkGxAxhwKLlGrgJFV9XFjBOwaVNSVHiSLBQPL+EAM3pVK5lJj56J0OwYbv5ZVjV2BOoDPQRdbxy63z8jW2tZE0RxhcYe2v3NkmLBUgPB6bU+8TNjYFObqWlM6cLEfh3zhWrwg4JYPg3BoUTdwntaNuA+97tQROVLXAr4KOT1dhH8/3iwkNxfeh/RVBymmBnPBilA22k0ozsWIDz/np2WCWUaobOLKzf3A5rsa62uO2uOZFgYyjDxALYQceOl1olMTnLJT9UDRjDUpzy3R5tRsPMU3jBqg75MWEMFTVJ4/Y5V1Qz2i8hJfSSAMR8hhOMOi1X0jKl6tjb8GTFWRVwz0Z6rkan0CtifuIFnkAOtkb2NbD9fl8FYGB24xWpVibLT1sdKHLX8FCydDyNHiJAaqwu/t8iM0aC3LoswvWAhbH6B9bx1ckRU1IC1vPEySNQGkOU78lDVVO6ZA4MgAJ5/l2VZ+Ga4bqfwXkdfZ77TxIPKgeeB+cN2d7QAiOpm7gxsgmGvJA6VZ3+fb3gC5v63HC/JvrUdo4weQfFevDka3wzWG6opCDn/bDQnhaFGxAjqEzwLVd1qIAlj0/77fRtx0gjHwyjQ9zQFusZQH8n3XQezGdRcdbq62ClbOQaYS/5wiyU6robRcpXXCRSETuZGFPGgeiCz6YxEvC3BTfkEBSWx6PeR1AXAqASGL/JR3/Xg64NpAS/5ZfMfKm3m89bbVcYvaW6/zVVKyWP7WaZVuuqBbd3oP7ISuZSSm/nQxrTdqZPSINuAt0xf32LqnGYZl+obp3AmM5rSB1G3XsE3HuwGOR4Hvp+m67+pO4Hi6tQ3SuUi1NduxbV03dS9wLc8zH9dS/JMYMW0fuPG9+/gIdNsHbgPD1t37OPG0IAB8gR0EAYDZweP6k38SJ7puBTdo3GLE8HTHMk3Tde5lxDA02/fswHFsU7d80zIf1+z8L3FiGq6l+05geffxYWuBFwASC5hxXT/wHtc3/W/xAYZnuoZn+Pcy4mu2Z9iBZYP2bM8PnMf1YHeGEdPQAhdYMNAMTd+3vMf1c3eHEUezwaJs27fMAAzMfFxreHf48LXAdgDQC4zAg+P0yDbzzjBimZptu7YfQLQJPNdwnMe1rHeHEQc0YhlgWhBGPF8PHtn93h1GAs22fINj0W048I/tpO8MJ7apQRTxwAWD73JM75E9+d3hwwWN2LrhYTwMHN60flR/f3c4CTT0WZ4ZGC6cFs91H/mtwM5w4lgapNC+7sCWuuWa/gMY2cnQ7rgQ2n070EEn8BA8JI/fydDu6liR6Ibt6aAWx3iIRnYytrsWRBLfdXwbQqLjmcYDGNnJ2O56ENtNOOQ6/HEM/SFuaydju6djbA8MCzNg1zIf8gXrTsZ2D6p2C86Gg2UiZPQPOCPmToZ2D4p2y7BN1wtM3cH/l/tDPnYytPtQspuu6Vl4jQKh/QFn3dzJyO5DzW6YnsEju2u7D0i2zJ0M7D7U7DpEEa4R33xIGDF3MrAHhuYHHl7lQfarGw9wvuZOxvXA0QAFbgoKgdTxIZa1k3E98DXfN3UI7ZgyWtZDNLKTcd3QTc3yTcfhhz140P2cGewmJw7qxMXLLS+AlOsBF6b6bjISaFBOgUZMCOy2/ZDreGsnI7thmKASF9NfvI/3Xf+OD7q7HaCmSfMXtZXwe+hvz779H7BfhGg='
for name,text in json.loads(zlib.decompress(base64.b64decode(RESEARCH_B64))).items():
 (WORK/name).write_text(text)
